In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, make_scorer, f1_score,
    precision_score, recall_score, roc_curve
)
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from scipy.stats import randint, uniform


import warnings
warnings.filterwarnings('ignore')

In [ ]:
import Models_for_Control

final_matrix = pd.read_parquet(r"MODEL_READY_MATRIX.parquet")
# make a new balanced dataset by randomly sampling controls to match the number of disease patients
RANDOM_SEEDS = [42, 69, 420, 80085]

disease = final_matrix[final_matrix['LABEL'] == 1]
control = final_matrix[final_matrix['LABEL'] == 0]

n_disease = len(disease)
n_control = len(control)

print(f"Disease patients: {n_disease}")
print(f"Control patients: {n_control}")
print(f"Imbalance ratio: 1:{n_control // n_disease}")

i = 0

for RANDOM_SEED in RANDOM_SEEDS:
    # random sample of controls, same size as disease cohort
    control_sampled = control.sample(n=n_disease, random_state=RANDOM_SEED)

    # concat and shuffle
    balanced_matrix = pd.concat([disease, control_sampled], ignore_index=True)
    balanced_matrix = balanced_matrix.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"\nBalanced matrix shape: {balanced_matrix.shape}")
    print(f"  Label=1 (Disease): {(balanced_matrix['LABEL']==1).sum()}")
    print(f"  Label=0 (Control): {(balanced_matrix['LABEL']==0).sum()}")

    # Train models on the balanced matrix
    lasso_cv, lasso_importance_df = Models_for_Control.LR(balanced_matrix)
    rf_cv, rf_tuned_cv, rf_tuned_importance_df = Models_for_Control.RF(balanced_matrix)
